# Interactive GUI for Prediction â€” ETL Pipeline

Uses the **ETL-trained models** (`_etl` suffix from `10_demand_etl_pipeline.ipynb`) and the **SQLite database** maintained by `etl_demand.py`.

- **Part 1 â€” Future Prediction**: forecast tomorrow's hourly demand; SMARD official day-ahead forecast shown for comparison.
- **Part 2 â€” Historical Comparison**: features, actual demand and SMARD forecast are loaded directly from the pre-computed DB view â€” no live API calls needed for historical dates.

| Aspect | 08 (legacy) | 11 (ETL) |
|---|---|---|
| Models | `*_bayesian.pkl` | `*_bayesian_etl.pkl` |
| Historical features | Re-fetched & re-engineered | Read from SQLite DB |
| Historical actuals | SMARD API (filter 410) | DB `energy_demand_mwh` |
| SMARD forecast (hist.) | SMARD API (filter 411) | DB `smard_forecast_mwh` |

## Setup: Imports, Database Update and Model Loading

- Imports all required libraries including `etl_demand.py` for DB access
- Calls `update_database()` once (idempotent â€” completes in seconds if already current)
- Loads the four ETL-trained models from `../models/`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

from datetime import date, timedelta, datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from fetch_demand_data import *
from train_predict_model import *
from etl_demand import update_database, get_connection, load_combined_data, prepare_for_prediction_tomorrow_etl

# â”€â”€ Consistent plot style â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
plt.rcParams.update({
    'axes.grid':      True,
    'grid.color':     '#DCDCDC',
    'grid.linewidth': 0.5,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'axes.facecolor': 'white',
    'font.family':    'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titlepad':  13,
    'axes.labelsize': 10,
    'axes.labelpad':  8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    True,
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#DCDCDC',
    'legend.framealpha': 1.0,
    'legend.fontsize':   9,
})

# Set European locale so date-picker calendars start on Monday
display(HTML('<script>document.documentElement.lang = "de-DE";</script>'))

# Ensure the database is current (idempotent â€” usually completes in seconds)
print("Checking / updating database...")
update_database()

# â”€â”€ Load ETL-trained models â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_MODEL_PATHS = {
    'LGBM':               '../models/best_lgbm_model_bayesian_etl.pkl',
    'LGBM_conservative':  '../models/best_lgbm_model_bayesian_conservative_etl.pkl',
    'XGBoost':            '../models/best_xgb_model_bayesian_etl.pkl',
    'XGBoost_conservative': '../models/best_xgb_model_bayesian_conservative_etl.pkl',
}
models = {name: load_model_from_pickle(path) for name, path in _MODEL_PATHS.items()}
print("Models loaded:", list(models.keys()))

Checking / updating database...
Database ready: c:\Data\Stackfuel-Unterricht\Portfolio\Europe_Electricity_Load\workspace_energy_demand\db\energy_demand.db

Current data status:
  energy         :  64672 rows | max: 2026-05-25T23:00:00+0200 | up-to-date: True
  weather        :  64825 rows | max: 2026-05-26T01:00:00+0200 | up-to-date: True

[Energy] Up to date â€” nothing to do.

[Weather] Up to date â€” nothing to do.

Done.
  energy_demand : 64672 rows | max: 2026-05-25T23:00:00+0200
  weather       : 64825 rows | max: 2026-05-26T01:00:00+0200
Models loaded: ['LGBM', 'LGBM_conservative', 'XGBoost', 'XGBoost_conservative']


---
# Part 1 â€” Future Prediction

- **Predict for Tomorrow** â€” forecast the full next day (00:00â€“23:00 Berlin time)

Tomorrow's features are built live: energy lag context is fetched from SMARD (last ~15 days) and weather forecast is fetched from the Open-Meteo API.  
Energy lag column names are remapped from the legacy `fetch_prepare_data` naming to the ETL DB naming so they match the feature set the model was trained on.

Results are shown as a line plot **and** an hourly table side-by-side.